# Taller 1 — Notebook 02: Limpieza y construcción de variables

**MINE-4101: Ciencia de Datos Aplicada · Supervisión de contratación pública de bienes**

---

## Objetivo del notebook

Aplicar de forma **reproducible y justificada** los tratamientos de calidad identificados en
`01_entendimiento`, y **construir las variables derivadas** necesarias para el análisis, en particular las
**tres banderas de desviación** que responden al requerimiento del negocio:

1. `adicion_plazo` — el contrato requirió **adiciones de plazo**.
2. `subejecucion` — el contrato **no ejecutó el presupuesto comprometido**.
3. `sin_liquidar` — el contrato **se cerró sin liquidar**.

Al final se exporta un dataset limpio (`data/secop_bienes_limpio.parquet`) que consume `03_analisis`.

> **Ejecutar después de** `01_entendimiento.ipynb`. Este notebook **sí** transforma los datos.
> **Criterio general:** no se eliminan filas salvo casos inválidos evidentes; los subconjuntos de análisis
> se marcan con **columnas booleanas** (`ciclo_cerrado`, `periodo_analisis`) para preservar trazabilidad.


In [1]:
# --- Imports y configuración ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

RUTA_ENTRADA = '../data/raw/secop_bienes.parquet'
RUTA_SALIDA = '../data/processed/secop_bienes_limpio.parquet'
RUTA_FIGS = '../figs'

df = pd.read_parquet(RUTA_ENTRADA)
n0 = len(df)
print(f'Filas iniciales: {n0:,}  |  Columnas: {df.shape[1]}')


Filas iniciales: 196,391  |  Columnas: 36


## 1. Verificaciones previas (encoding y duplicados)

En `01` se verificó que las tildes/ñ de las categorías están correctas en UTF-8 (el `�` de consola es un
artefacto de visualización, **no** un problema de datos). Lo reconfirmamos y revisamos duplicados de `id_contrato`.


In [2]:
# Reconfirmar encoding correcto (code points reales)
ejemplo = df['modalidad_de_contratacion'].value_counts().index[0]
print('Ejemplo categoría:', ejemplo, '->', [hex(ord(c)) for c in ejemplo][:6], '...')
print('Contiene í (U+00ED):', 'í' in ejemplo, '-> NO requiere corrección de encoding')

# Duplicados
dups = df['id_contrato'].duplicated().sum()
print(f'\nid_contrato duplicados: {dups:,}')
if dups > 0:
    df = df.drop_duplicates(subset='id_contrato', keep='first').reset_index(drop=True)
    print(f'-> deduplicado. Filas: {len(df):,}')


Ejemplo categoría: Mínima cuantía -> ['0x4d', '0xed', '0x6e', '0x69', '0x6d', '0x61'] ...
Contiene í (U+00ED): True -> NO requiere corrección de encoding

id_contrato duplicados: 0


## 2. Limpieza de variables numéricas

### 2.1 `valor_del_contrato`: valores no válidos y outliers extremos

- Valores **≤ 0** no son contratos válidos → se marcan como nulos (`valor_del_contrato_clean`).
- El máximo (~1,3e16 COP) es implausible → se aplica **winsorización superior en el P99.9** para métricas
  sensibles a la escala, conservando siempre el valor original y una versión en **log**.


In [3]:
# valor_del_contrato limpio: <=0 -> NaN
df['valor_del_contrato_clean'] = df['valor_del_contrato'].where(df['valor_del_contrato'] > 0, np.nan)
n_invalidos = df['valor_del_contrato_clean'].isna().sum()

# Winsorización superior al P99.9
p999 = df['valor_del_contrato_clean'].quantile(0.999)
df['valor_del_contrato_w'] = df['valor_del_contrato_clean'].clip(upper=p999)
n_winsor = (df['valor_del_contrato_clean'] > p999).sum()

# Versión log (para análisis y visualización)
df['valor_log'] = np.log10(df['valor_del_contrato_clean'])

print(f'valor <= 0 marcados como nulo: {n_invalidos:,}')
print(f'Umbral P99.9: {p999:,.0f} COP  |  contratos winsorizados: {n_winsor:,}')
print(df[['valor_del_contrato', 'valor_del_contrato_w']].describe([.5, .99]).round(0).to_string())


valor <= 0 marcados como nulo: 188
Umbral P99.9: 31,000,990,678 COP  |  contratos winsorizados: 197


             valor_del_contrato  valor_del_contrato_w
count                196,391.00            196,203.00
mean          65,824,255,028.00        259,283,047.00
std       29,015,352,967,837.00      1,459,547,428.00
min                        0.00                  0.00
50%               33,600,000.00         33,683,540.00
99%            4,118,481,943.00      4,120,000,000.00
max   12,858,450,316,000,000.00     31,000,990,678.00


### 2.2 `dias_adicionados`: outliers implausibles

El máximo (~14.276 días ≈ 39 años) es implausible. Para la **magnitud** se topa a **1.095 días (3 años)**;
la **bandera binaria** (`> 0`) no se ve afectada por el tope.


In [4]:
TOPE_DIAS = 365 * 3  # 1.095 días
n_tope = (df['dias_adicionados'] > TOPE_DIAS).sum()
df['dias_adicionados_clean'] = df['dias_adicionados'].clip(upper=TOPE_DIAS)
print(f'Contratos con dias_adicionados > {TOPE_DIAS} (topados): {n_tope:,}')
print(f'Máximo original: {df["dias_adicionados"].max():,}  ->  tras tope: {df["dias_adicionados_clean"].max():,}')


Contratos con dias_adicionados > 1095 (topados): 1
Máximo original: 14,276  ->  tras tope: 1,095


### 2.3 Inconsistencias en valores de ejecución

- `valor_pendiente_de_pago` **< 0** es inconsistente → se marca (no se usa como positivo).
- `valor_pagado` **>** `valor_del_contrato` indica **sobre-ejecución** → se marca; el ratio de ejecución se
  acota a [0, 1] para las métricas.


In [5]:
df['pendiente_negativo'] = df['valor_pendiente_de_pago'] < 0
df['sobre_ejecucion'] = df['valor_pagado'] > df['valor_del_contrato']
print(f'valor_pendiente_de_pago < 0: {df["pendiente_negativo"].sum():,}')
print(f'valor_pagado > valor_del_contrato (sobre-ejecución): {df["sobre_ejecucion"].sum():,}')


valor_pendiente_de_pago < 0: 144
valor_pagado > valor_del_contrato (sobre-ejecución): 177


## 3. Parseo de `duraci_n_del_contrato` a días

Viene como texto con unidades heterogéneas ("245 Dia(s)", "5 Mes(es)", "3 Semana(s)", "2 Año(s)",
"No definido"). Se extrae el número y la unidad y se convierte a **días**. Casos sin número o "No definido"
quedan como nulo.


In [6]:
import re

FACTORES = {'hora': 1/24, 'dia': 1, 'semana': 7, 'mes': 30, 'año': 365, 'ano': 365}

def a_dias(txt):
    if not isinstance(txt, str):
        return np.nan
    s = txt.strip().lower()
    m = re.search(r'(\d+(?:[.,]\d+)?)', s)
    if m is None:
        return np.nan
    num = float(m.group(1).replace(',', '.'))
    for clave, factor in FACTORES.items():
        if clave in s:
            return num * factor
    return np.nan

df['duracion_dias'] = df['duraci_n_del_contrato'].apply(a_dias)
print('Parseo de duración a días:')
print(f'  válidos: {df["duracion_dias"].notna().sum():,}  |  nulos: {df["duracion_dias"].isna().sum():,}')
print(df['duracion_dias'].describe([.25, .5, .75, .95]).round(1).to_string())


Parseo de duración a días:
  válidos: 175,572  |  nulos: 20,819
count       175,572.00
mean          6,961.50
std       2,218,513.50
min               0.00
25%              24.00
50%              60.00
75%             139.00
95%             300.00
max     902,760,750.00


## 4. Fechas y año de firma

- Se crea `anio_firma` a partir de `fecha_de_firma`.
- `fecha_de_inicio_del_contrato` nula (~1,2%) → se **imputa** con `fecha_de_firma` para no perder registros
  en cálculos de duración (decisión conservadora: firma ≈ inicio).


In [7]:
df['anio_firma'] = df['fecha_de_firma'].dt.year

n_inicio_nulo = df['fecha_de_inicio_del_contrato'].isna().sum()
df['fecha_inicio_imputada'] = df['fecha_de_inicio_del_contrato'].isna()
df['fecha_de_inicio_del_contrato'] = df['fecha_de_inicio_del_contrato'].fillna(df['fecha_de_firma'])
print(f'fecha_de_inicio imputada con fecha_de_firma: {n_inicio_nulo:,} registros')
print('Rango de años de firma:', df['anio_firma'].min(), '-', df['anio_firma'].max())


fecha_de_inicio imputada con fecha_de_firma: 2,440 registros
Rango de años de firma: 2019 - 2025


## 5. Marcado del ciclo y periodo de análisis

Según la justificación de `01`:

- **`ciclo_cerrado`**: contratos en estado `terminado` o `Cerrado` — únicos con ejecución completable, por
  tanto los válidos para medir **subejecución** y **no-liquidación**.
- **`periodo_analisis`**: contratos con firma **≤ 2023**. Los firmados en 2024–2025 están fuertemente
  **censurados a la derecha** (mayor `% en ejecución`, ciclo aún abierto), por lo que se excluyen de las
  desviaciones que requieren cierre. Las **adiciones de plazo** sí pueden analizarse en la ventana completa.


In [8]:
ANIO_CORTE = 2023
df['ciclo_cerrado'] = df['estado_contrato'].isin(['terminado', 'Cerrado'])
df['periodo_analisis'] = df['anio_firma'] <= ANIO_CORTE

print(f'Corte de periodo para subejecución/liquidación: firma <= {ANIO_CORTE}')
print(f'  ciclo_cerrado (terminado/Cerrado): {df["ciclo_cerrado"].sum():,} ({df["ciclo_cerrado"].mean()*100:.1f}%)')
print(f'  periodo_analisis (firma <= {ANIO_CORTE}): {df["periodo_analisis"].sum():,} ({df["periodo_analisis"].mean()*100:.1f}%)')
print(f'  base subejec/liquidación (ciclo_cerrado & periodo): {(df["ciclo_cerrado"] & df["periodo_analisis"]).sum():,}')


Corte de periodo para subejecución/liquidación: firma <= 2023
  ciclo_cerrado (terminado/Cerrado): 101,054 (51.5%)
  periodo_analisis (firma <= 2023): 121,644 (61.9%)
  base subejec/liquidación (ciclo_cerrado & periodo): 68,351


## 6. Construcción de las tres banderas de desviación

### 6.1 `adicion_plazo`
Adición de plazo si `dias_adicionados > 0`. Aplica a todos los contratos (observable sin cierre).


In [9]:
df['adicion_plazo'] = df['dias_adicionados'] > 0
print(f'adicion_plazo: {df["adicion_plazo"].sum():,} contratos ({df["adicion_plazo"].mean()*100:.1f}%)')


adicion_plazo: 18,671 contratos (9.5%)


### 6.2 `subejecucion`

"No ejecutar el presupuesto comprometido". Por el **subreporte de `valor_pagado`** (detectado en `01`),
usamos `valor_facturado` (mejor cobertura) y calculamos el **ratio de ejecución** acotado a [0, 1]:

`ratio_ejecucion = min(valor_facturado, valor_del_contrato) / valor_del_contrato`

La bandera solo se define sobre contratos **con ciclo cerrado, en el periodo, con valor válido y con ejecución
reportada** (`valor_facturado > 0`). Los contratos cerrados que reportan ejecución **0** se apartan como
`sin_ejecucion_reportada` (problema de captura, no subejecución genuina), para no sesgar el análisis. Umbral:
`ratio_ejecucion < 0.9` (se reporta sensibilidad en `03`).


In [10]:
UMBRAL_SUBEJEC = 0.9

# Ratio de ejecución acotado a [0,1] usando valor_facturado
val = df['valor_del_contrato_clean']
df['ratio_ejecucion'] = (np.minimum(df['valor_facturado'], val) / val).clip(0, 1)

# Base elegible: ciclo cerrado, en periodo y con valor válido
base = df['ciclo_cerrado'] & df['periodo_analisis'] & val.notna()

# Contratos cerrados sin ejecución reportada (facturado == 0): problema de captura
df['sin_ejecucion_reportada'] = base & (df['valor_facturado'] == 0)

# Bandera de subejecución solo donde hay ejecución reportada
elegible_sub = base & (df['valor_facturado'] > 0)
df['subejecucion'] = np.where(elegible_sub, df['ratio_ejecucion'] < UMBRAL_SUBEJEC, np.nan)

print(f'Base elegible (ciclo cerrado & periodo & valor válido): {base.sum():,}')
print(f'  - sin ejecución reportada (facturado=0, apartados): {df["sin_ejecucion_reportada"].sum():,}')
print(f'  - con ejecución reportada (base de la bandera): {elegible_sub.sum():,}')
print(f'subejecucion = True: {np.nansum(df["subejecucion"]):,.0f}  '
      f'({np.nanmean(df["subejecucion"])*100:.1f}% de los elegibles)')


Base elegible (ciclo cerrado & periodo & valor válido): 68,229
  - sin ejecución reportada (facturado=0, apartados): 30,101
  - con ejecución reportada (base de la bandera): 38,128
subejecucion = True: 2,994  (7.9% de los elegibles)


### 6.3 `sin_liquidar`

"Cerrarse sin liquidar". Definida sobre contratos con **ciclo cerrado en el periodo**:
`sin_liquidar = (liquidaci_n == 'No')`. En estados en ejecución la no-liquidación es esperable, por lo que
allí la bandera queda indefinida (nula).


In [11]:
base_liq = df['ciclo_cerrado'] & df['periodo_analisis']
df['sin_liquidar'] = np.where(base_liq, df['liquidaci_n'].str.strip().str.lower() == 'no', np.nan)
print(f'Base (ciclo cerrado & periodo): {base_liq.sum():,}')
print(f'sin_liquidar = True: {np.nansum(df["sin_liquidar"]):,.0f}  '
      f'({np.nanmean(df["sin_liquidar"])*100:.1f}% de la base)')


Base (ciclo cerrado & periodo): 68,351


sin_liquidar = True: 40,431  (59.2% de la base)


### 6.4 Resumen de las banderas y solapamiento

In [12]:
# Prevalencia de cada bandera
resumen_flags = pd.DataFrame({
    'bandera': ['adicion_plazo', 'subejecucion', 'sin_liquidar'],
    'base': [len(df), int((df['subejecucion'].notna()).sum()), int((df['sin_liquidar'].notna()).sum())],
    'n_desviados': [int(df['adicion_plazo'].sum()), int(np.nansum(df['subejecucion'])), int(np.nansum(df['sin_liquidar']))],
})
resumen_flags['tasa_%'] = (resumen_flags['n_desviados'] / resumen_flags['base'] * 100).round(1)
print(resumen_flags.to_string(index=False))

# Solapamiento entre banderas (sobre la base común: ciclo cerrado & periodo)
com = df[base_liq].copy()
com['f_adic'] = com['adicion_plazo'].astype(int)
com['f_sub'] = com['subejecucion'].fillna(False).astype(int)
com['f_liq'] = com['sin_liquidar'].fillna(False).astype(int)
com['n_desv'] = com[['f_adic', 'f_sub', 'f_liq']].sum(axis=1)
print('\nContratos por nº de desviaciones simultáneas (base cerrada & periodo):')
print(com['n_desv'].value_counts().sort_index().to_string())


      bandera   base  n_desviados  tasa_%
adicion_plazo 196391        18671    9.50
 subejecucion  38128         2994    7.90
 sin_liquidar  68351        40431   59.20



Contratos por nº de desviaciones simultáneas (base cerrada & periodo):
n_desv
0    26368
1    40328
2     1648
3        7


## 7. Exportación del dataset limpio

Se guarda el dataset con las columnas originales + las derivadas y banderas. `03_analisis` lo consume y filtra
según `ciclo_cerrado` / `periodo_analisis` según la desviación a estudiar.


In [13]:
cols_derivadas = [
    'valor_del_contrato_clean', 'valor_del_contrato_w', 'valor_log', 'dias_adicionados_clean',
    'duracion_dias', 'anio_firma', 'ratio_ejecucion',
    'ciclo_cerrado', 'periodo_analisis', 'fecha_inicio_imputada',
    'pendiente_negativo', 'sobre_ejecucion', 'sin_ejecucion_reportada',
    'adicion_plazo', 'subejecucion', 'sin_liquidar',
]
print('Columnas derivadas añadidas:', len(cols_derivadas))
print(f'Dataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas')

df.to_parquet(RUTA_SALIDA, index=False)
print(f'\nGuardado en: {RUTA_SALIDA}')


Columnas derivadas añadidas: 16
Dataset final: 196,391 filas x 52 columnas



Guardado en: ../data/processed/secop_bienes_limpio.parquet


## 8. Síntesis de la limpieza

| Tratamiento | Resultado |
|-------------|-----------|
| Encoding | Verificado correcto (UTF-8); sin corrección necesaria |
| `id_contrato` duplicados | 0 detectados |
| `valor_del_contrato` ≤ 0 | Marcados como nulos (`valor_del_contrato_clean`) |
| Outliers de valor | Winsorización P99.9 (`valor_del_contrato_w`) + `valor_log` |
| `dias_adicionados` implausibles | Topados a 1.095 días (`dias_adicionados_clean`) |
| Inconsistencias de ejecución | Marcadas (`pendiente_negativo`, `sobre_ejecucion`) |
| `duraci_n_del_contrato` | Parseada a `duracion_dias` |
| `fecha_de_inicio` nula | Imputada con `fecha_de_firma` (marca `fecha_inicio_imputada`) |
| Periodo | `periodo_analisis` = firma ≤ 2023; `ciclo_cerrado` = terminado/Cerrado |
| **Banderas de desviación** | `adicion_plazo`, `subejecucion`, `sin_liquidar` construidas |
| Subreporte de ejecución | Contratos cerrados con facturado=0 apartados (`sin_ejecucion_reportada`) |

**Siguiente paso:** `03_analisis.ipynb` — estadística, pruebas de hipótesis y visualización multivariada
sobre `data/secop_bienes_limpio.parquet`.
